# BusState Data Processing Pipeline (Python)

### Overview

This notebook processes raw **BusState APC data** from compressed `.txt.zip` files into clean, structured datasets for downstream analysis and dashboarding.

The workflow is a Python translation of an existing R-based pipeline previously used in the OSU Department of Transportation and Traffic Management (TTM), with improvements for modularity, readability, and scalability.

---

### Data Source

* Directory: `K:/AP/TTM/Data/APC Data/`
* File format: `compressed .txt`
* Naming convention:

  ```
  busstate0####DDMMYY.txt.zip
  ```

  * `####` = bus identifier
  * `DD` = day
  * `MM` = month
  * `YY` = year (2-digit)

---

### Output (`sort_and_save`)

* Data is split into monthly subsets

* Saved as `.csv` files:

  ```
  YYYY-MMM-busstate.csv
  ```

* Example:

  ```
  2025-OCT-busstate.csv
  ```

* Output directory:

  ```
  K:/AP/TTM/Data/WMC Dashboard/BusState Cleaned/
  ```

---

### Notes

* This will take around 3-4 minutes to run for one month worth of data.
* If this is run on a Unix based machine, changes will need to be made to the root directory.
* Potential issues with overwriting in the sort_and_save function when running a new month with existing data. - will revisit

---

### Author

* Writen by Clayton Morgan (morgan.1461) Reporting and Analytics Analyst at The Ohio State University
* Python implementation of legacy R workflow in the TTM


In [1]:
import busstate_processing as bp
import dashboard_processing as dp
# note for future me: if you edit processing.py, run the lines below to reload w/out restarting kernel

# import importlib
# importlib.reload(bp)
# importlib.reload(dp)

**Note**: The year and month fields must be in 2 digit format, within the quotations.

e.g) For October, 2025

year = "25"

month = "09"

In [2]:
# STEP 1: Set month and year of interest for dashboard
year = "25"
month = "12"

In [3]:
# STEP 2: Run processing pipeline to clean busstate data for month and year of interest
bp.busstate_processing(year, month)

Starting busstate processing for 12/25...
Found 902 busstate files for 12/25 in 'K:/AP/TTM/Data/APC Data'
Unzipping and processing busstate files for 12/25...
Combined dataframe has 868244 records for 12/25
No data for month JAN 2025
No data for month FEB 2025
No data for month MAR 2025
No data for month APR 2025
No data for month MAY 2025
No data for month JUN 2025
No data for month JUL 2025
No data for month AUG 2025
Saved 9 records for SEP 2025 to 'K:\AP\TTM\Data\WMC Dashboard\BusState Cleaned\2025-SEP-busstate.csv'
No data for month OCT 2025
Saved 26107 records for NOV 2025 to 'K:\AP\TTM\Data\WMC Dashboard\BusState Cleaned\2025-NOV-busstate.csv'
Saved 842127 records for DEC 2025 to 'K:\AP\TTM\Data\WMC Dashboard\BusState Cleaned\2025-DEC-busstate.csv'
Finished processing busstate data for 12/25 in 180.94 seconds. Cleaned files saved to 'K:/AP/TTM/Data/WMC Dashboard/BusState Cleaned'


### Stops inventory and metrics
This code will produce a dataframe containing the stops including the new University Hopsital and Doan Hall stops.

In [30]:
import pandas as pd
import numpy as np
import os
import pathlib

In [18]:
def build_stops_df():
    '''
    Build stops dataframe by merging static pattern stops and stop inventory files. Anytime that stops change, this will need to be re run after the csv files are updated.

    Parameters:
        None
    Returns:
        stops_df (pd.DataFrame): dataframe with stop id, stop name, and lat/lon coordinates for all stops in the pattern stops file
    '''
    # set directory paths for stop data - stored as 2 csv files in ./stops/
    stop_data_dir = pathlib.Path(os.getcwd()) / "stops"
    pattern_stops_path = stop_data_dir / "pattern_stops.csv"
    stop_inventory_path = stop_data_dir / "stop_inventory.csv"

    # read in static stop files
    pattern_stops = pd.read_csv(pattern_stops_path, header = None) # no header
    stop_inventory = pd.read_csv(stop_inventory_path)

    # only need cols 0 and 9 and can drop any dups
    pattern_stops = pattern_stops.iloc[:, [0, 9]].drop_duplicates()
    pattern_stops.columns = ["ROUTE", "STOP_ID"] # rename cols for merge

    # Merge pattern stops and stop inventory on stop id - left merge 
    stops_df = pattern_stops.merge(stop_inventory, how = "left", on = "STOP_ID")

    return stops_df

In [27]:
stops_df = build_stops_df()

stops_df[stops_df['STOP_ID'] == 37]

,ROUTE,STOP_ID,STOP_NAME,TIMEPOINT_NAME,LONG,LAT,HEADING
75,MC,37,DOAN HALL,DOAN,-83.017457,39.996147,90.0


In [101]:
def which_stop(lat, lon, route = "MC", max_distance = 0.0005):
    '''
    Determine the closest stop to a given latitude and longitude. Med center route is default, but can be updated to other routes as needed. 
    NOTE: stops_df must be a global variable for this function to work, so build_stops_df() must be run before this function is called.

    Args:
        lat (float): The latitude of the point of interest.
        lon (float): The longitude of the point of interest.
        route (str): The route to filter stops by. Default is "MC" for medical center. Based on ROUTE col in stops DataFrame. Potentially important in future for overlapping stops.
        max_distance (float): The maximum distance to consider for a stop. Default is 0.0005 per legacy R code. approximately 150ish feet

    Returns:
        int: The STOP_ID of the closest stop.
    '''
    # subset the stops to route of interest
    route_stops = stops_df[stops_df['ROUTE'] == route]

    distance_lon = route_stops['LONG'].values - lon
    distance_lat = route_stops['LAT'].values - lat

    # calculate the distance to each stop
    distances = np.sqrt(distance_lon**2 + distance_lat**2)

    mask = distances < max_distance
    # lowest distance should be the closest stop now. - each stop is at minimum approx 430ft apart so margin of 150 should be fine for MC route.
    selected_stop = route_stops[mask]

    # if no stops within alloted distance, return None
    if not np.any(mask):
        return None

    return int(route_stops.loc[mask, 'STOP_ID'].iloc[0])


In [102]:
lat = 39.995880
lon = -83.0204271

s = which_stop(lat, lon)
s
# tested with points on google maps, works within 150 foot radius of stop.

401

In [103]:
# def process_mc_busstate():
#     """
#     Process the busstate data for the medical center route.
#     NOTE: This will ONLY work for the medical center route.
#     NOTE: This will return a significantly smaller dataframe
#     Args:
#         busstate_df (DataFrame): A DataFrame containing busstate data from the cleaned busstate files.
#         stops_df (DataFrame): A DataFrame containing stop information, including 'STOP_ID', 'LAT', and 'LONG' columns.
#         stop_inventory (DataFrame): A DataFrame containing stop inventory information.
#     Returns:
#         DataFrame: A pandas DataFrame containing the processed busstate data for the medical center route.
#     """
year_full = "2025" # add function to convert, etc
month_full = "DEC"

busstate_dir = pathlib.Path(os.getcwd()) / "BusState Cleaned"
busstate_path = os.path.normpath(os.path.join(busstate_dir, f"{year_full}-{month_full}-busstate.csv"))
busstate_df = pd.read_csv(busstate_path) # Now cleaned busstate data read in

In [108]:
# Process the busstate data for medical center routes
#busstate_df.info()

# should filter to just MC routes
filtered_busstate_df = busstate_df.loc[(busstate_df['RUN_ID'] > 1500) & (busstate_df['RUN_ID'] < 1600)].copy()

# Convert time metrics to datetime
filtered_busstate_df['EVENT_TIME'] = pd.to_datetime(filtered_busstate_df['EVENT_TIME'], format = "%H:%M:%S")
filtered_busstate_df['DEPARTURE_TIME'] = pd.to_datetime(filtered_busstate_df['DEPARTURE_TIME'], format = "%H:%M:%S")
filtered_busstate_df['ENTER_STOP_WINDOW_TIME'] = pd.to_datetime(filtered_busstate_df['ENTER_STOP_WINDOW_TIME'], format = "%H:%M:%S")
filtered_busstate_df['EXIT_STOP_WINDOW_TIME'] = pd.to_datetime(filtered_busstate_df['EXIT_STOP_WINDOW_TIME'], format = "%H:%M:%S")

# sort by date and event time
filtered_busstate_df = filtered_busstate_df.sort_values(['DATE', 'EVENT_TIME'])

# Assign stop ID - most resource intensive step - as INT not float
filtered_busstate_df['STOP_ID'] = filtered_busstate_df.apply(lambda row: which_stop(row['LATITUDE'], row['LONGITUDE']), axis = 1)

In [ ]:
# filter out 'dummy' stops, 27 and 461
filtered_busstate_df = filtered_busstate_df.drop(filtered_busstate_df[filtered_busstate_df['STOP_ID'].isin([27, 461])].index)

# filter out empty stops - where bus was on route and not at a stop geographically
filtered_busstate_df = filtered_busstate_df.drop(filtered_busstate_df[filtered_busstate_df['STOP_ID'].isna()].index)

# Convert to int
filtered_busstate_df['STOP_ID'] = filtered_busstate_df['STOP_ID'].astype(int)

# filter by bus id, date, event time
filtered_busstate_df = filtered_busstate_df.sort_values(['BUS_ID', 'DATE', 'EVENT_TIME']).reset_index(drop = True) # restting index

In [122]:
filtered_busstate_df

,DATE,BUS_ID,RUN_ID,DEST_SIGN_ROUTE_TEXT,BLOCK_ID,TRIP_ID,ROUTE_ID,STOP_SEQUENCE,LATITUDE,LONGITUDE,...,EVENT_TYPE,EVENT_TIME,BOARDINGS,ALIGHTINGS,PASSENGER_LOAD,TRIP_START_TIME,DEPARTURE_TIME,ENTER_STOP_WINDOW_TIME,EXIT_STOP_WINDOW_TIME,STOP_ID
0,2025-12-01,1301,1512.0,H008,23688702.0,NaN,NaN,-1,40.001041,-83.037102,...,7,1900-01-01 05:50:04,0,0,0,NaN,NaT,NaT,NaT,403
1,2025-12-01,1301,1512.0,MC,23688702.0,NaN,MC21,-1,40.001045,-83.037079,...,6,1900-01-01 05:50:05,0,0,0,NaN,NaT,NaT,NaT,403
2,2025-12-01,1301,1512.0,MC,23688702.0,3746020.0,MC21,-1,40.001045,-83.037079,...,15,1900-01-01 05:50:05,0,0,0,05:50:04,NaT,NaT,NaT,403
3,2025-12-01,1301,1512.0,MC,23688702.0,3746020.0,MC21,-1,40.001045,-83.037079,...,9,1900-01-01 05:50:05,0,0,0,05:50:04,NaT,NaT,NaT,403
4,2025-12-01,1301,1512.0,MC,23688702.0,3746020.0,MC21,-1,40.001045,-83.037079,...,10,1900-01-01 05:50:05,0,0,0,05:50:04,NaT,NaT,NaT,403
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117778,2025-12-30,2503,1507.0,MC,23690702.0,571020.0,MC25,3,40.002815,-83.042542,...,3,1900-01-01 19:41:58,0,2,0,19:37:43,1900-01-01 19:42:04,1900-01-01 19:41:50,1900-01-01 19:42:17,94
117779,2025-12-30,2503,1507.0,MC,23690702.0,571020.0,MC25,4,40.002899,-83.041107,...,4,1900-01-01 19:43:17,0,0,0,19:37:43,1900-01-01 19:43:17,1900-01-01 19:43:04,1900-01-01 19:43:31,95
117780,2025-12-30,2503,1507.0,MC,23690702.0,571020.0,MC25,5,39.995872,-83.020683,...,4,1900-01-01 19:50:27,0,0,0,19:37:43,1900-01-01 19:50:27,1900-01-01 19:50:21,1900-01-01 19:50:33,401
117781,2025-12-30,2503,1507.0,MC,23690702.0,571020.0,MC25,6,39.996178,-83.017693,...,3,1900-01-01 19:51:22,0,0,0,19:37:43,1900-01-01 19:51:22,1900-01-01 19:51:07,1900-01-01 19:51:37,37
